# Module: Informed Search

Informed search uses an estimate of how far each node is from the goal.

| # | Algorithm | Frontier ordered by |
|---|---|---|
| 1 | Greedy Best-First Search | h(n) |
| 2 | A* Search | f(n) = g(n) + h(n) |

Graphs use the same adjacency dict as before. The heuristic is a second dict: `h[n]` estimates the
cost from n to the goal. Each function returns `(path, cost, nodes_expanded)`.

## Heuristics

A heuristic **h(n)** estimates the cost of the cheapest path from n to a goal, using only the state
itself (for example, straight-line distance on a map). **h\*(n)** is the true cost.

**Admissible:** h never overestimates.

$$0 \le h(n) \le h^*(n) \quad \text{for every node } n$$

**Consistent:** for every edge from n to n' with cost c(n, n'),

$$h(n) \le c(n, n') + h(n')$$

and h(goal) = 0. It is a triangle inequality: one step can lower the estimate by at most the cost
of that step.

**Consistency is the stronger condition.**

- Consistent implies admissible. Apply the inequality along an optimal path
  n → n<sub>1</sub> → … → goal; the terms add up to h(n) ≤ h\*(n).
- Admissible does not imply consistent. Take A → B → G with c(A, B) = 1 and c(B, G) = 3, so
  h\*(A) = 4 and h\*(B) = 3. The values h(A) = 4 and h(B) = 1 are admissible, but 4 > 1 + 1.
- With a consistent h, f = g + h never decreases along a path. A* then has the optimal g for a
  node the first time it pops it and never has to reopen it. With an h that is only admissible,
  graph-search A* may need to reopen nodes.

All heuristics in this notebook are consistent.

<h2><b>1. Greedy Best-First Search</b></h2>

![](data\greedy_best_first.gif)

GIF source: <TODO>

Greedy best-first expands the node that looks closest to the goal: the one with the smallest h(n).
It ignores what it has already paid to get there.

The frontier is a priority queue (`heapq`) ordered by h(n), plus an `explored` set. UCS from the
previous notebook orders by the cost paid so far, g(n); greedy orders by the estimated cost left.
It usually reaches the goal in few expansions, but a node with a small h can sit behind an
expensive edge.

**Step-by-step mechanism of Greedy Best-First Search:**

- Push `(h(start), [start])` onto the priority queue; `explored` is empty.
- Loop while the queue is not empty:
  - Pop the path whose last node has the smallest h; let `node` be that node.
  - If `node` is the goal, return the path (**goal found**).
  - If `node` is already explored, skip it.
  - Mark `node` as explored and count it as expanded.
  - Push `(h(neighbor), path + [neighbor])` for every neighbour not yet explored.
- If the queue becomes empty, return **failure**.

**Properties of Greedy Best-First Search**

| Complete? | Optimal? | Time | Space |
|---|---|---|---|
| Yes on finite graphs; no on infinite ones | No | O(b<sup>m</sup>) | O(b<sup>m</sup>) |

- **Complete?** The explored set prevents loops on a finite graph; in an infinite space a misleading h can lead it down an endless path.
- **Optimal?** It never looks at g(n), so it takes expensive edges towards nodes that only look close.
- **Time:** with a useless h it explores everything down to depth m; a good h cuts that sharply.
- **Space:** it keeps every generated node in the frontier.

**Case study.** Sari is late for her flight. At each junction she drives towards the place whose
signboard shows the shortest straight-line distance to the **Airport**. `h` holds those distances
in km; edge costs are driving distances in km.

In [1]:
city_roads = {
    'Home':    {'Market': 4, 'School': 3},
    'Market':  {'Home': 4, 'Bridge': 6, 'Mall': 2},
    'School':  {'Home': 3, 'Mall': 3},
    'Bridge':  {'Market': 6, 'Tunnel': 7, 'Station': 2},
    'Mall':    {'Market': 2, 'School': 3, 'Station': 3},
    'Tunnel':  {'Bridge': 7, 'Airport': 3},
    'Station': {'Bridge': 2, 'Mall': 3, 'Airport': 4},
    'Airport': {'Tunnel': 3, 'Station': 4},
}

h_airport = {
    'Home': 9, 'Market': 6, 'School': 7, 'Bridge': 4,
    'Mall': 5, 'Tunnel': 2, 'Station': 3, 'Airport': 0,
}

In [2]:
import heapq

def greedy_best_first(graph, h, start, goal, verbose=False):
    """Greedy best-first graph search. Returns (path, cost, nodes_expanded)."""
    frontier = [(h[start], [start])]  # priority queue ordered by h(n)
    explored = set()
    nodes_expanded = 0

    while frontier:
        h_value, path = heapq.heappop(frontier)
        node = path[-1]
        if node == goal:
            cost = sum(graph[u][v] for u, v in zip(path, path[1:]))
            return path, cost, nodes_expanded
        if node in explored:
            continue
        explored.add(node)
        nodes_expanded += 1
        if verbose:
            print(f"expand {node:<8} h = {h_value}")
        for neighbor in graph[node]:
            if neighbor not in explored:
                heapq.heappush(frontier, (h[neighbor], path + [neighbor]))

    return None, float('inf'), nodes_expanded

In [3]:
path, cost, expanded = greedy_best_first(city_roads, h_airport, 'Home', 'Airport', verbose=True)
print()
print("Path found     :", " -> ".join(path))
print("Driving km     :", cost)
print("Nodes expanded :", expanded)

expand Home     h = 9
expand Market   h = 6
expand Bridge   h = 4
expand Tunnel   h = 2

Path found     : Home -> Market -> Bridge -> Tunnel -> Airport
Driving km     : 20
Nodes expanded : 4


Greedy expands 4 nodes but takes the Bridge and the Tunnel because they look close.
Home → Market → Mall → Station → Airport is 13 km.

<h2><b>2. A* Search</b></h2>

![](data\a_star.gif)

GIF source: <TODO>

A* expands the node with the smallest f(n) = g(n) + h(n): cost paid plus estimated cost left, an
estimate of the best solution that passes through n.

Like UCS it keeps a priority queue, a table of best g values, and a goal test on pop. Compared with
greedy, g(n) in the priority stops A* from chasing cheap-looking nodes behind expensive edges. With
an admissible h (tree search) or a consistent h (graph search), A* returns the cheapest path. With
h = 0 it is UCS.

**Step-by-step mechanism of A\*:**

- Push `(f = h(start), g = 0, [start])` onto the priority queue and set `best_g[start] = 0`.
- Loop while the queue is not empty:
  - Pop the entry with the smallest f; let `node` be the last node of its path.
  - If its g is larger than `best_g[node]`, the entry is stale: skip it.
  - If `node` is the goal, return the path and g (**goal found**).
  - Count `node` as expanded. For each neighbour, compute `new_g = g + step_cost`.
  - If `new_g` beats `best_g[neighbor]`, record it and push
    `(new_g + h(neighbor), new_g, path + [neighbor])`.
- If the queue becomes empty, return **failure**.

**Properties of A\***

| Complete? | Optimal? | Time | Space |
|---|---|---|---|
| Yes (finite b, every step cost ≥ ε > 0) | Yes, with admissible h (tree search) or consistent h (graph search) | O(b<sup>d</sup>) worst case | O(b<sup>d</sup>) |

- **Complete?** f grows by at least ε per step, so only finitely many nodes have f ≤ C*, and A* expands those first.
- **Optimal?** An admissible h never makes the optimal path look more expensive than C*, so A* cannot pop the goal through a worse path first.
- **Time:** h = 0 gives UCS; the closer h is to h\*, the less A* expands, but most practical heuristics stay exponential in d.
- **Space:** A* keeps every generated node, and memory usually limits it before time does.

**Case study.** An ambulance takes a patient from an **Accident** site to the **Hospital**. Edge
costs are driving minutes. `h` is straight-line distance divided by the ambulance's top speed, so
it never overestimates.

In [4]:
ambulance_map = {
    'Accident': {'Plaza': 3, 'Library': 4},
    'Plaza':    {'Accident': 3, 'Bank': 5, 'Park': 7},
    'Library':  {'Accident': 4, 'Bank': 2},
    'Bank':     {'Plaza': 5, 'Library': 2, 'Harbor': 4, 'Park': 2},
    'Park':     {'Plaza': 7, 'Bank': 2, 'Hospital': 6},
    'Harbor':   {'Bank': 4, 'Hospital': 3},
    'Hospital': {'Park': 6, 'Harbor': 3},
}

h_hospital = {
    'Accident': 10, 'Plaza': 8, 'Library': 7, 'Bank': 5,
    'Park': 4, 'Harbor': 3, 'Hospital': 0,
}

In [5]:
import heapq

def a_star(graph, h, start, goal, verbose=False):
    """A* graph search. Returns (path, cost, nodes_expanded)."""
    frontier = [(h[start], 0, [start])]   # (f, g, path)
    best_g = {start: 0}
    nodes_expanded = 0

    while frontier:
        f, g, path = heapq.heappop(frontier)
        node = path[-1]
        if g > best_g[node]:              # stale entry
            continue
        if node == goal:
            return path, g, nodes_expanded
        nodes_expanded += 1
        if verbose:
            print(f"expand {node:<9} g = {g:>2}  h = {h[node]:>2}  f = {f:>2}")
        for neighbor, step in graph[node].items():
            new_g = g + step
            if new_g < best_g.get(neighbor, float('inf')):
                best_g[neighbor] = new_g
                heapq.heappush(frontier, (new_g + h[neighbor], new_g, path + [neighbor]))

    return None, float('inf'), nodes_expanded

In [6]:
path, cost, expanded = a_star(ambulance_map, h_hospital, 'Accident', 'Hospital', verbose=True)
print()
print("Path found     :", " -> ".join(path))
print("Minutes        :", cost)
print("Nodes expanded :", expanded)

expand Accident  g =  0  h = 10  f = 10
expand Plaza     g =  3  h =  8  f = 11
expand Library   g =  4  h =  7  f = 11
expand Bank      g =  6  h =  5  f = 11
expand Park      g =  8  h =  4  f = 12
expand Harbor    g = 10  h =  3  f = 13

Path found     : Accident -> Library -> Bank -> Harbor -> Hospital
Minutes        : 13
Nodes expanded : 6


## Comparison

Both algorithms on the same graph and heuristic: a courier going from city **A** to city **H**.

In [7]:
courier_map = {
    'A': {'B': 4, 'C': 3},
    'B': {'A': 4, 'D': 3, 'E': 10},
    'C': {'A': 3, 'F': 3},
    'D': {'B': 3, 'G': 4},
    'E': {'B': 10, 'H': 4},
    'F': {'C': 3, 'G': 6},
    'G': {'D': 4, 'F': 6, 'H': 3},
    'H': {'E': 4, 'G': 3},
}

h_courier = {'A': 10, 'B': 7, 'C': 8, 'D': 5, 'E': 3, 'F': 6, 'G': 2, 'H': 0}

In [8]:
def print_table(rows):
    header = f"{'algorithm':<22} | {'path':<40} | {'cost':>5} | {'nodes expanded':>14}"
    print(header)
    print("-" * len(header))
    for name, path, cost, expanded in rows:
        path_str = " -> ".join(path) if path else "no solution"
        print(f"{name:<22} | {path_str:<40} | {cost:>5} | {expanded:>14}")

rows = []
for name, algorithm in [('Greedy Best-First', greedy_best_first), ('A*', a_star)]:
    path, cost, expanded = algorithm(courier_map, h_courier, 'A', 'H')
    rows.append((name, path, cost, expanded))
print_table(rows)

algorithm              | path                                     |  cost | nodes expanded
------------------------------------------------------------------------------------------
Greedy Best-First      | A -> B -> E -> H                         |    18 |              3
A*                     | A -> B -> D -> G -> H                    |    14 |              6


Greedy reaches H in 3 expansions because E has the smallest h (3), but the edge B → E costs 10, so
its route costs 18. A* expands 6 nodes, accounts for the cost of reaching E, and returns
A → B → D → G → H at cost 14. Both use the same heuristic; the only difference is g(n) in the
priority.